# Лабораторная работа 1 — рабочий ноутбук

**ФИО:**
**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = 'data/S15/clients.csv'
RAW_TXN = 'data/S15/transactions.csv'

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [56]:
raw_clients = pd.read_csv(
    RAW_CLIENTS,
    sep=',',
    decimal='.',
    encoding='utf-8',
    dtype={'branch_code': str}
)

raw_transactions = pd.read_csv(
    RAW_TXN,
    sep=';',
    decimal=',',
    encoding='cp1251'
)

display(raw_clients.head(2))
display(raw_transactions.head(2))

,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C001919,1968-10-29,Ж,Санкт-Петербург,Массовый,Базовый,Дебетовая,14.06.2011,13.02.2016,Зарплата в нашем банке,53500.0,Да,2024-10,NaN,00100,Да
1,C002920,1954-10-07,М,СПб,Массовый,Золото,Кредитная,02.04.2012,16.10.2016,Зарплата в другом банке,125500.0,Да,2024-03,2.0,00120,Да


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990124,C000052,2023-11-26 15:27:11,5482.85,RUB,Одежда и обувь,5651,Карта,успешно
1,990022,C002010,2023-12-10 11:38:58,1606.39,RUB,Аптеки,5912,СБП,успешно


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [57]:
display(pd.DataFrame([
    {
        'Таблица': 'clients',
        'Единица наблюдения': 'Клиент (физлицо)',
        'Строк (N)': len(raw_clients),
        'Колонок': raw_clients.shape[1],
        'Первичный ключ': 'client_id',
        'Уникальных ID (U)': raw_clients['client_id'].nunique(),
        'Лишних записей ключа': len(raw_clients) - raw_clients['client_id'].nunique(),
        'Доля небиективности': f"{(len(raw_clients) - raw_clients['client_id'].nunique()) / len(raw_clients):.2%}"
    },
    {
        'Таблица': 'transactions',
        'Единица наблюдения': 'Транзакция по карте',
        'Строк (N)': len(raw_transactions),
        'Колонок': raw_transactions.shape[1],
        'Первичный ключ': 'txn_id',
        'Уникальных ID (U)': raw_transactions['txn_id'].nunique(),
        'Лишних записей ключа': len(raw_transactions) - raw_transactions['txn_id'].nunique(),
        'Доля небиективности': f"{(len(raw_transactions) - raw_transactions['txn_id'].nunique()) / len(raw_transactions):.2%}"
    }
]))

types_c = pd.DataFrame({'Колонка clients': raw_clients.columns, 'Тип': raw_clients.dtypes.values.astype(str)})
types_t = pd.DataFrame({'Колонка transactions': raw_transactions.columns, 'Тип': raw_transactions.dtypes.values.astype(str)})
display(pd.concat([types_c, types_t], axis=1).fillna(''))

,Таблица,Единица наблюдения,Строк (N),Колонок,Первичный ключ,Уникальных ID (U),Лишних записей ключа,Доля небиективности
0,clients,Клиент (физлицо),5670,16,client_id,5615,55,0.97%
1,transactions,Транзакция по карте,619916,9,txn_id,618416,1500,0.24%


,Колонка clients,Тип,Колонка transactions,Тип
0,client_id,str,txn_id,int64
1,birth_date,str,client_id,str
2,gender,str,txn_datetime,str
3,city,str,amount,float64
4,segment,str,currency,str
5,loyalty_tier,str,category,str
6,card_type,str,mcc_code,int64
7,registration_date,str,channel,str
8,card_open_date,str,status,str
9,income_source,str,,


### 2.2. Дубли

In [58]:
def audit_duplicates(df, key_col, table_name):
    n_total = len(df)
    n_clones = df.duplicated().sum()
    
    df_dedup = df.drop_duplicates()
    conflict_mask = df_dedup.duplicated(subset=[key_col], keep=False)
    
    conflict_ids = df_dedup.loc[conflict_mask, key_col].unique()
    n_conflict_keys = len(conflict_ids)
    n_conflict_rows = conflict_mask.sum()
    
    return {
        'Таблица': table_name,
        'Строк (N)': n_total,
        'Ключ': key_col,
        'Полные клоны (строки)': n_clones,
        'Доля клонов': f"{n_clones / n_total:.2%}",
        'Конфликтных ID (ключей)': n_conflict_keys,
        'Строк с конфликтами': n_conflict_rows,
        'Доля конфликтных строк': f"{n_conflict_rows / n_total:.2%}"
    }, conflict_ids, df_dedup

res_c, conf_ids_c, dedup_c = audit_duplicates(raw_clients, 'client_id', 'clients')
res_t, conf_ids_t, dedup_t = audit_duplicates(raw_transactions, 'txn_id', 'transactions')

display(pd.DataFrame([res_c, res_t]))

print(f"\n--- Примеры реальных конфликтов в clients (всего {len(conf_ids_c)} проблемных ID) ---")
display(dedup_c[dedup_c['client_id'].isin(conf_ids_c[:3])].sort_values('client_id'))

print(f"\n--- Примеры реальных конфликтов в transactions (всего {len(conf_ids_t)} проблемных ID) ---")
display(dedup_t[dedup_t['txn_id'].isin(conf_ids_t[:3])].sort_values('txn_id'))

,Таблица,Строк (N),Ключ,Полные клоны (строки),Доля клонов,Конфликтных ID (ключей),Строк с конфликтами,Доля конфликтных строк
0,clients,5670,client_id,40,0.71%,15,30,0.53%
1,transactions,619916,txn_id,1266,0.20%,234,468,0.08%



--- Примеры реальных конфликтов в clients (всего 15 проблемных ID) ---


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
478,C000221,1956-01-05,М,Казань,Массовый,Базовый,Дебетовая,20.09.2015,07.07.2018,Зарплата в другом банке,42200.0,Нет,NaN,2.0,00445,Нет
4400,C000221,1956-01-05,М,Санкт-Петербург,Премиальный,Базовый,Дебетовая,20.09.2015,07.07.2018,Зарплата в другом банке,42200.0,Нет,NaN,2.0,00445,Нет
407,C002745,1968-03-15,Ж,Нижний Новгород,Премиальный,Базовый,Кредитная,24.03.2023,25.02.2025,Не указан,81600.0,Да,2024-04,NaN,00128,Да
3007,C002745,1968-03-15,Ж,Екатеринбург,Массовый,Базовый,Кредитная,24.03.2023,25.02.2025,Не указан,81600.0,Да,2024-04,NaN,00128,Да
340,C005196,1962-06-21,М,Москва,Массовый,Базовый,Дебетовая,28.05.2020,17.11.2022,Зарплата в другом банке,80700.0,Нет,NaN,NaN,00698,Да
1108,C005196,1962-06-21,М,Санкт-Петербург,Массовый,Базовый,Дебетовая,28.05.2020,17.11.2022,Зарплата в другом банке,80700.0,Нет,NaN,NaN,00698,Да



--- Примеры реальных конфликтов в transactions (всего 234 проблемных ID) ---


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
2787,102780,C005545,2024-01-05 05:51:33,561.09,RUB,Кафе и рестораны,5812,Карта,успешно
2788,102780,C005545,2024-01-05,561.09,RUB,Кафе и рестораны,5812,Карта,успешно
6956,106938,C005104,2024-01-08 11:57:04,9321.67,RUB,кафе и рестораны,5812,Онлайн-банк,успешно
6957,106938,C005104,2024-01-08,9321.67,RUB,кафе и рестораны,5812,Онлайн-банк,успешно
9546,109521,C001309,2024-01-11 19:51:49,1324.05,RUB,Аптеки,5912,Карта,успешно
9547,109521,C001309,2024-01-11,1324.05,RUB,Аптеки,5912,Карта,успешно


### 2.3. Соответствие данных описанию полей из задания

In [67]:
missing_records = []
for name, df in [('clients', raw_clients), ('transactions', raw_transactions)]:
    null_counts = df.isna().sum()
    for col in df.columns:
        if null_counts[col] > 0:
            missing_records.append({
                'Таблица': name,
                'Колонка': col,
                'Пропусков (шт)': null_counts[col],
                'Доля от N': f"{null_counts[col] / len(df):.2%}"
            })

print("1. ПРОПУСКИ В ДАННЫХ:")
display(pd.DataFrame(missing_records))


num_records = [
    {
        'Поле': f"{tbl}.{col}",
        'Min': s.min(),
        'Max': s.max(),
        'Значений < 0': (s < 0).sum(),
        'Значений == 0': (s == 0).sum()
    }
    for tbl, col, s in [
        ('clients', 'monthly_income', raw_clients['monthly_income']),
        ('clients', 'satisfaction_score', raw_clients['satisfaction_score']),
        ('transactions', 'amount', raw_transactions['amount'])
    ]
]
print("\n2. ЧИСЛОВЫЕ МЕТРИКИ:")
display(pd.DataFrame(num_records))


neg_df = raw_transactions[raw_transactions['amount'] < 0]
neg_summary = neg_df['status'].value_counts().reset_index()
neg_summary.columns = ['Статус при amount < 0', 'Количество (шт)']
neg_summary['Доля'] = (neg_summary['Количество (шт)'] / len(neg_df)).map('{:.2%}'.format)

print("\nПрирода отрицательных сумм (amount < 0):")
display(neg_summary)


cat_cols = [
    ('clients.segment', 4, raw_clients['segment']),
    ('clients.loyalty_tier', 4, raw_clients['loyalty_tier']),
    ('clients.card_type', 3, raw_clients['card_type']),
    ('clients.gender', 2, raw_clients['gender']),
    ('clients.has_mobile_app', 2, raw_clients['has_mobile_app']),
    ('clients.marketing_opt_in', 2, raw_clients['marketing_opt_in']),          
    ('clients.income_source', 'Источники дохода', raw_clients['income_source']), 
    ('clients.branch_code', 'Коды отделений', raw_clients['branch_code']),  
    ('clients.city', 'Города РФ', raw_clients['city']),
    ('transactions.category', 9, raw_transactions['category']),
    ('transactions.channel', 4, raw_transactions['channel']),
    ('transactions.status', 3, raw_transactions['status']),
    ('transactions.currency', 'Валюта', raw_transactions['currency']),
    ('transactions.mcc_code', 'MCC-коды', raw_transactions['mcc_code']),   
]

cat_records = [
    {
        'Поле': field,
        'Ожидалось по ТЗ': expected,
        'Факт (уник.)': len(s.dropna().unique()),
        'Все фактические значения': ", ".join(sorted([str(x) for x in s.dropna().unique()]))
    }
    for field, expected, s in cat_cols
]
print("\n3. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ:")
display(pd.DataFrame(cat_records))

print("\nДетализация ненормализованных полей:")
display(raw_transactions['category'].value_counts().reset_index().rename(columns={'category': 'Категория (12 шт вместо 9)', 'count': 'Операций'}))
display(raw_clients['city'].value_counts().reset_index().rename(columns={'city': 'Город (17 вариантов вместо 6)', 'count': 'Клиентов'}))


t_dates = pd.to_datetime(raw_transactions['txn_datetime'], format='mixed')
c_birth = pd.to_datetime(raw_clients['birth_date'], format='mixed', dayfirst=True)
c_reg = pd.to_datetime(raw_clients['registration_date'], format='mixed', dayfirst=True)
c_card = pd.to_datetime(raw_clients['card_open_date'], format='mixed', dayfirst=True)

out_of_period_cnt = ((t_dates < '2024-01-01') | (t_dates > '2025-06-30 23:59:59')).sum()
underage_cnt = (((pd.to_datetime('2025-06-30') - c_birth).dt.days // 365.25) < 18).sum()
card_before_reg_cnt = (c_card < c_reg).sum()

date_audit = pd.DataFrame([
    {
        'Сущность / Поле': 'transactions.txn_datetime (период ТЗ: 2024-01-01 .. 2025-06-30)',
        'Минимум в данных': str(t_dates.min()),
        'Максимум в данных': str(t_dates.max()),
        'Аномалии / Нарушения': f"{out_of_period_cnt} транзакций вне периода ТЗ"
    },
    {
        'Сущность / Поле': 'clients.Возраст (на дату среза 30.06.2025)',
        'Минимум в данных': f"{((pd.to_datetime('2025-06-30') - c_birth).dt.days // 365.25).min():.0f} лет",
        'Максимум в данных': f"{((pd.to_datetime('2025-06-30') - c_birth).dt.days // 365.25).max():.0f} лет",
        'Аномалии / Нарушения': f"{underage_cnt} клиентов младше 18 лет"
    },
    {
        'Сущность / Поле': 'clients.Логика: карта открыта раньше регистрации',
        'Минимум в данных': '-',
        'Максимум в данных': '-',
        'Аномалии / Нарушения': f"{card_before_reg_cnt} нарушений"
    }
])

print("\n4. АУДИТ ДАТ И ВРЕМЕННЫХ ГРАНИЦ:")
display(date_audit)

1. ПРОПУСКИ В ДАННЫХ:


,Таблица,Колонка,Пропусков (шт),Доля от N
0,clients,monthly_income,532,9.38%
1,clients,app_last_login,2103,37.09%
2,clients,satisfaction_score,3359,59.24%



2. ЧИСЛОВЫЕ МЕТРИКИ:


,Поле,Min,Max,Значений < 0,Значений == 0
0,clients.monthly_income,-999.00,1052900.00,56,0
1,clients.satisfaction_score,1.00,5.00,0,0
2,transactions.amount,-60058.31,152959.72,7285,0



Природа отрицательных сумм (amount < 0):


,Статус при amount < 0,Количество (шт),Доля
0,возврат,7285,100.00%


... (3 outputs left out)

,Категория (12 шт вместо 9),Операций
0,Продукты,163184
1,Одежда и обувь,69646
2,Транспорт,64952
3,АЗС,57029
4,Кафе и рестораны,48267
5,Развлечения и досуг,47527
6,Аптеки,45313
7,Онлайн-сервисы,43426
8,Связь и интернет,29270
9,Связь и Интернет,19131


,Город (17 вариантов вместо 6),Клиентов
0,Москва,1055
1,Нижний Новгород,769
2,Санкт-Петербург,637
3,Новосибирск,536
4,Екатеринбург,506
5,Казань,489
6,Н. Новгород,213
7,МОСКВА,181
8,москва,181
9,г. Москва,180



4. АУДИТ ДАТ И ВРЕМЕННЫХ ГРАНИЦ:


,Сущность / Поле,Минимум в данных,Максимум в данных,Аномалии / Нарушения
0,transactions.txn_datetime (период ТЗ: 2024-01-...,2023-11-26 15:27:11,2026-07-28 16:55:46,82 транзакций вне периода ТЗ
1,clients.Возраст (на дату среза 30.06.2025),19 лет,89 лет,0 клиентов младше 18 лет
2,clients.Логика: карта открыта раньше регистрации,-,-,0 нарушений


### 2.4. Связь между таблицами

In [60]:
c_ids = set(raw_clients['client_id'].dropna())
t_ids = set(raw_transactions['client_id'].dropna())

orphan_tx = raw_transactions['client_id'].isin(t_ids - c_ids).sum()

display(pd.DataFrame([
    {'Метрика': 'Уникальных клиентов в clients', 'Кол-во': len(c_ids), 'Доля от N_clients': '100.00%'},
    {'Метрика': 'Клиентов с операциями', 'Кол-во': len(c_ids & t_ids), 'Доля от N_clients': f"{len(c_ids & t_ids) / len(c_ids):.2%}"},
    {'Метрика': 'Клиентов без операций', 'Кол-во': len(c_ids - t_ids), 'Доля от N_clients': f"{len(c_ids - t_ids) / len(c_ids):.2%}"},
    {'Метрика': 'Транзакций без клиента', 'Кол-во': orphan_tx, 'Доля от N_transactions': f"{orphan_tx / len(raw_transactions):.2%}"}
]))

,Метрика,Кол-во,Доля от N_clients,Доля от N_transactions
0,Уникальных клиентов в clients,5615,100.00%,NaN
1,Клиентов с операциями,5611,99.93%,NaN
2,Клиентов без операций,4,0.07%,NaN
3,Транзакций без клиента,30647,NaN,4.94%


**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

1. **Ключи и дубли:**
   - `clients`: 40 полных клонов (0.71% от 5 670 строк) и 15 коллизий ID (30 строк, 0.53%). Конфликтные ID удаляем целиком — не понятно какие из них верные, вводят в заблуждение.
   - `transactions`: 1 266 клонов (0.20% от 619 916 строк) и 234 коллизии `txn_id` (468 строк, 0.08%).
2. 30 647 транзакций (4.94% от 619 916) без связи с `clients`. Исключаем — клиентов по ним идентифицировать нельзя, поэтому к данному анализу не подходят.
3. **Период:** транзакции вне ТЗ (`2024-01-01 .. 2025-06-30`). Отсекаем.
4. **Числовые аномалии:**
   - `monthly_income == -999.00` (56 строк, 0.99% от 5 670) — технический маркер пропуска, заменяем на `NaN`.
   - `amount < 0` (7 285 строк, 1.18% от 619 916) — 100% операции со статусом «возврат». Оставляем с минусом для корректного сальдо трат.
5. **Текстовая грязь:** 17 вариантов написания городов вместо базовых и 12 категорий вместо 9 (ошибки регистра и разделителей). Требуется нормализация.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [61]:
log_records = []

df_clients = raw_clients.copy()
df_transactions = raw_transactions.copy()

mask_income_sentinel = df_clients['monthly_income'] == -999.0
n_income_sentinels = int(mask_income_sentinel.sum())

df_clients.loc[mask_income_sentinel, 'monthly_income'] = np.nan

log_records.append({
    'field': 'monthly_income',
    'issue': 'системный маркер пропуска данных (-999)',
    'scope': 'клиенты со значением -999',
    'n_rows': n_income_sentinels,
    'value_before': '-999.00',
    'action': 'исправлено',
    'value_after': 'NaN',
    'rationale': 'технический маркер пропущенного значения, заменен на стандартный np.nan'
})

client_date_cols = ['birth_date', 'registration_date', 'card_open_date']
for col in client_date_cols:
    df_clients[col] = pd.to_datetime(df_clients[col], format='mixed')

df_transactions['txn_datetime'] = pd.to_datetime(df_transactions['txn_datetime'], format='mixed')

df_transactions['mcc_code'] = df_transactions['mcc_code'].astype(str)

print(f"Заменено маркеров -999 в доходе: {n_income_sentinels} шт.\n")
print("Clients (проверка дат и дохода):")
display(df_clients[['client_id'] + client_date_cols + ['monthly_income']].head(3))

print("\nTransactions (проверка даты и mcc_code):")
display(df_transactions[['txn_id', 'txn_datetime', 'mcc_code']].head(3))

Заменено маркеров -999 в доходе: 56 шт.

Clients (проверка дат и дохода):


,client_id,birth_date,registration_date,card_open_date,monthly_income
0,C001919,1968-10-29,2011-06-14,2016-02-13,53500.0
1,C002920,1954-10-07,2012-02-04,2016-10-16,125500.0
2,C000925,1973-02-04,2014-04-18,2016-05-27,69700.0



Transactions (проверка даты и mcc_code):


,txn_id,txn_datetime,mcc_code
0,990124,2023-11-26 15:27:11,5651
1,990022,2023-12-10 11:38:58,5912
2,100001,2024-01-01 00:54:00,4111


### 3.2. Категориальные признаки

In [62]:
df_clients['city'] = df_clients['city'].astype(str).str.strip()

city_map = {
    'МОСКВА': 'Москва', 'москва': 'Москва', 'г. Москва': 'Москва', 'Москва': 'Москва',
    'Санкт-Петербург': 'Санкт-Петербург', 'Санкт Петербург': 'Санкт-Петербург', 
    'СПб': 'Санкт-Петербург', 'г. Санкт-Петербург': 'Санкт-Петербург',
    'Нижний Новгород': 'Нижний Новгород', 'Н. Новгород': 'Нижний Новгород',
    'Новосибирск': 'Новосибирск', 'г. Новосибирск': 'Новосибирск',
    'Екатеринбург': 'Екатеринбург', 'ЕКАТЕРИНБУРГ': 'Екатеринбург', 'г.Екатеринбург': 'Екатеринбург',
    'Казань': 'Казань', 'казань': 'Казань'
}

n_cities_before = raw_clients['city'].nunique()
mask_city_changed = raw_clients['city'] != raw_clients['city'].str.strip().replace(city_map)
n_city_rows = int(mask_city_changed.sum())

df_clients['city'] = df_clients['city'].replace(city_map)

log_records.append({
    'field': 'city',
    'issue': 'разнобой написания (регистр, префиксы г., сокращения СПб, Н. Новгород)',
    'scope': 'все строки clients',
    'n_rows': n_city_rows,
    'value_before': '17 вариантов написания (г. Москва, СПб, казань и др.)',
    'action': 'исправлено',
    'value_after': f'{df_clients["city"].nunique()} канонических городов РФ',
    'rationale': 'сведено к единому классификатору 6 городов РФ без потери исходного смысла'
})

category_map = {
    'Кафе/рестораны': 'Кафе и рестораны',
    'кафе и рестораны': 'Кафе и рестораны',
    'Связь и Интернет': 'Связь и интернет'
}

mask_cat_changed = raw_transactions['category'].isin(category_map.keys())
n_cat_rows = int(mask_cat_changed.sum())

df_transactions['category'] = df_transactions['category'].replace(category_map)

log_records.append({
    'field': 'category',
    'issue': '12 вариантов вместо 9 по ТЗ (разнобой регистра и разделителей / и «и»)',
    'scope': 'все строки transactions',
    'n_rows': n_cat_rows,
    'value_before': '12 категорий (Кафе/рестораны, Связь и Интернет и др.)',
    'action': 'исправлено',
    'value_after': f'{df_transactions["category"].nunique()} категорий',
    'rationale': 'приведено к 9 регламентированным категориям ТЗ в соответствии с кодами MCC'
})

print(f"Города: было {n_cities_before} -> стало {df_clients['city'].nunique()}:")
print(sorted(df_clients['city'].dropna().unique()))

print(f"\nКатегории: было 12 -> стало {df_transactions['category'].nunique()}:")
print(sorted(df_transactions['category'].dropna().unique()))

Города: было 17 -> стало 6:
['Екатеринбург', 'Казань', 'Москва', 'Нижний Новгород', 'Новосибирск', 'Санкт-Петербург']

Категории: было 12 -> стало 9:
['АЗС', 'Аптеки', 'Кафе и рестораны', 'Одежда и обувь', 'Онлайн-сервисы', 'Продукты', 'Развлечения и досуг', 'Связь и интернет', 'Транспорт']


### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

In [63]:
app_cross = pd.crosstab(
    df_clients['has_mobile_app'], 
    df_clients['app_last_login'].isna(), 
    rownames=['Наличие приложения'], 
    colnames=['Пропуск в дате входа (True=NaN)']
)
print("Связь наличия приложения и пропусков даты входа:")
display(app_cross)

n_clients_total = len(df_clients)

missing_summary = pd.DataFrame([
    {
        'Поле': 'app_last_login',
        'Пропусков (шт)': df_clients['app_last_login'].isna().sum(),
        'Доля': f"{df_clients['app_last_login'].isna().sum() / n_clients_total:.2%}",
        'Механизм': 'Структурный / MAR',
        'Обоснование': 'Клиенты без приложения (has_mobile_app == Нет)',
        'Решение': 'Оставлено NaN (заполнение создаст ложных пользователей)'
    },
    {
        'Поле': 'satisfaction_score',
        'Пропусков (шт)': df_clients['satisfaction_score'].isna().sum(),
        'Доля': f"{df_clients['satisfaction_score'].isna().sum() / n_clients_total:.2%}",
        'Механизм': 'MNAR (ошибка отклика)',
        'Обоснование': 'Опрос добровольный, порядковая шкала 1-5',
        'Решение': 'Оставлено NaN (импутация исказит оценку удовлетворенности)'
    },
    {
        'Поле': 'monthly_income',
        'Пропусков (шт)': df_clients['monthly_income'].isna().sum(),
        'Доля': f"{df_clients['monthly_income'].isna().sum() / n_clients_total:.2%}",
        'Механизм': 'MNAR (сокрытие дохода)',
        'Обоснование': 'Отказ называть доход при регистрации (+ маркеры -999)',
        'Решение': 'Оставлено NaN (сохраняем исходное распределение доходов)'
    }
])

print(f"\nСводная таблица пропусков в clients (Всего строк N = {n_clients_total}):")
display(missing_summary)

Связь наличия приложения и пропусков даты входа:


Пропуск в дате входа (True=NaN),False,True
Наличие приложения,,
Да,3567,0
Нет,0,2103



Сводная таблица пропусков в clients (Всего строк N = 5670):


,Поле,Пропусков (шт),Доля,Механизм,Обоснование,Решение
0,app_last_login,2103,37.09%,Структурный / MAR,Клиенты без приложения (has_mobile_app == Нет),Оставлено NaN (заполнение создаст ложных польз...
1,satisfaction_score,3359,59.24%,MNAR (ошибка отклика),"Опрос добровольный, порядковая шкала 1-5",Оставлено NaN (импутация исказит оценку удовле...
2,monthly_income,588,10.37%,MNAR (сокрытие дохода),Отказ называть доход при регистрации (+ маркер...,Оставлено NaN (сохраняем исходное распределени...


### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

In [64]:
def find_iqr_outliers(series, name):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    outliers = s[s > upper_fence]
    
    return {
        'Признак': name,
        'N (без NaN)': len(s),
        'Медиана': f"{s.median():,.2f}",
        'Q1 (25%)': f"{q1:,.2f}",
        'Q3 (75%)': f"{q3:,.2f}",
        'Граница МКР (верх)': f"{upper_fence:,.2f}",
        '99-й перцентиль': f"{s.quantile(0.99):,.2f}",
        'Максимум': f"{s.max():,.2f}",
        'Выбросов > границы': len(outliers),
        'Доля от N': f"{len(outliers) / len(s):.2%}"
    }, upper_fence

stat_inc, inc_fence = find_iqr_outliers(df_clients['monthly_income'], 'monthly_income (доход)')

stat_amt, amt_fence = find_iqr_outliers(df_transactions[df_transactions['amount'] > 0]['amount'], 'amount (покупки > 0)')

print("1. МАТЕМАТИЧЕСКИЕ ГРАНИЦЫ ВЫБРОСОВ (МКР / IQR):")
display(pd.DataFrame([stat_inc, stat_amt]))

rich_clients = df_clients[df_clients['monthly_income'] > inc_fence]
print(f"\nСегменты клиентов с доходом выше границы МКР ({inc_fence:,.0f} ₽):")
display(rich_clients['segment'].value_counts(dropna=False).to_frame().T)

log_records.append({
    'field': 'monthly_income',
    'issue': 'значения выше верхней границы МКР (>197,050 ₽)',
    'scope': f'клиенты с доходом > {inc_fence:,.0f} ₽',
    'n_rows': stat_inc['Выбросов > границы'],
    'value_before': f'диапазон до {stat_inc["Максимум"]} ₽',
    'action': 'оставлено',
    'value_after': f'диапазон до {stat_inc["Максимум"]} ₽',
    'rationale': '96% выбросов относятся к сегментам Премиальный и Private; отражают естественное распределение состоятельных клиентов'
})

log_records.append({
    'field': 'amount',
    'issue': 'суммы операций выше границы МКР (>5,014 ₽)',
    'scope': f'операции с суммой > {amt_fence:,.0f} ₽',
    'n_rows': stat_amt['Выбросов > границы'],
    'value_before': f'диапазон до {stat_amt["Максимум"]} ₽',
    'action': 'оставлено',
    'value_after': f'диапазон до {stat_amt["Максимум"]} ₽',
    'rationale': 'крупные покупки (электроника, путешествия) являются легитимным потребительским поведением; удаление исказит оборот клиентов'
})

1. МАТЕМАТИЧЕСКИЕ ГРАНИЦЫ ВЫБРОСОВ (МКР / IQR):


,Признак,N (без NaN),Медиана,Q1 (25%),Q3 (75%),Граница МКР (верх),99-й перцентиль,Максимум,Выбросов > границы,Доля от N
0,monthly_income (доход),5082,"75,000.00","52,300.00","110,200.00","197,050.00","462,752.00","1,052,900.00",380,7.48%
1,amount (покупки > 0),612631,"1,089.52",486.12,"2,297.28","5,014.03","13,050.77","152,959.72",50338,8.22%



Сегменты клиентов с доходом выше границы МКР (197,050 ₽):


segment,Премиальный,Private,Зарплатный,Массовый
count,185,180,9,6


### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

In [68]:
n_c_raw = len(df_clients)

df_c_step1 = df_clients.drop_duplicates()
n_c_clones = n_c_raw - len(df_c_step1)

conflicting_client_ids = df_c_step1[df_c_step1.duplicated(subset=['client_id'], keep=False)]['client_id'].unique()
clean_clients = df_c_step1[~df_c_step1['client_id'].isin(conflicting_client_ids)].copy().reset_index(drop=True)
n_c_conflicts = len(df_c_step1) - len(clean_clients)

log_records.append({
    'field': 'all (строки clients)',
    'issue': 'полные дубликаты записей',
    'scope': 'все клиенты',
    'n_rows': int(n_c_clones),
    'value_before': 'полные клоны строк',
    'action': 'удалено',
    'value_after': 'уникальные строки',
    'rationale': 'устранение технического сбоя повторной выгрузки'
})

log_records.append({
    'field': 'client_id',
    'issue': 'коллизии первичного ключа (конфликт атрибутов под одним ID)',
    'scope': f"client_id: {', '.join(conflicting_client_ids[:3])} и др.",
    'n_rows': int(n_c_conflicts),
    'value_before': '30 строк с противоречивыми данными',
    'action': 'удалено',
    'value_after': 'ключи исключены',
    'rationale': 'невозможно достоверно определить истинные данные клиента'
})

funnel_clients = pd.DataFrame([
    {'Шаг фильтрации': '0. Исходная база клиентов', 'Исключено (шт)': 0, 'Осталось (шт)': n_c_raw, 'Доля от исх.': '100.00%'},
    {'Шаг фильтрации': '1. Удаление полных дубликатов', 'Исключено (шт)': n_c_clones, 'Осталось (шт)': len(df_c_step1), 'Доля от исх.': f"{len(df_c_step1) / n_c_raw:.2%}"},
    {'Шаг фильтрации': '2. Исключение конфликтующих ID', 'Исключено (шт)': n_c_conflicts, 'Осталось (шт)': len(clean_clients), 'Доля от исх.': f"{len(clean_clients) / n_c_raw:.2%}"}
])

print("ВОРОНКА ФИЛЬТРАЦИИ КЛИЕНТОВ (CLIENTS):")
display(funnel_clients)

n_t_raw = len(df_transactions)

has_exact_time = (df_transactions['txn_datetime'].dt.time != pd.to_datetime('00:00:00').time()).astype(int)
df_t_step1 = (
    df_transactions.assign(_has_time=has_exact_time)
    .sort_values(by=['txn_id', '_has_time'], ascending=[True, False])
    .drop_duplicates(subset=['txn_id'], keep='first')
    .drop(columns=['_has_time'])
)
n_t_dedup = n_t_raw - len(df_t_step1)


mask_period = (df_t_step1['txn_datetime'] >= '2024-01-01') & (df_t_step1['txn_datetime'] <= '2025-06-30 23:59:59')
df_t_step2 = df_t_step1[mask_period].copy()
n_t_period = len(df_t_step1) - len(df_t_step2)


mask_known_client = df_t_step2['client_id'].isin(clean_clients['client_id'])
clean_transactions = df_t_step2[mask_known_client].copy().reset_index(drop=True)
n_t_orphans = len(df_t_step2) - len(clean_transactions)

log_records.append({
    'field': 'txn_id',
    'issue': 'дубликаты txn_id (полные клоны и записи с обрезанным временем)',
    'scope': 'все транзакции',
    'n_rows': int(n_t_dedup),
    'value_before': 'повторяющиеся txn_id',
    'action': 'удалено',
    'value_after': 'уникальные txn_id с полным временем',
    'rationale': 'сохранена более детальная запись с секундами, дубли удалены'
})

log_records.append({
    'field': 'txn_datetime',
    'issue': 'операции вне периода ТЗ (<2024-01-01 и >2025-06-30)',
    'scope': 'транзакции конца 2023 г. и 2025-2026 гг.',
    'n_rows': int(n_t_period),
    'value_before': '2023-11 .. 2026-07',
    'action': 'ограничено',
    'value_after': '2024-01-01 .. 2025-06-30',
    'rationale': 'строгое соответствие границам анализа по регламенту ТЗ'
})

log_records.append({
    'field': 'client_id (transactions)',
    'issue': 'транзакции без профиля в clients (сироты)',
    'scope': 'операции по неизвестным ID',
    'n_rows': int(n_t_orphans),
    'value_before': 'транзакции без клиента',
    'action': 'удалено',
    'value_after': 'только клиенты из чистой базы',
    'rationale': 'клиенты не идентифицированы, согласие на маркетинг неизвестно'
})

funnel_tx = pd.DataFrame([
    {'Шаг фильтрации': '0. Исходная база операций', 'Исключено (шт)': 0, 'Осталось (шт)': n_t_raw, 'Доля от исх.': '100.00%'},
    {'Шаг фильтрации': '1. Дедупликация txn_id (с сохранением времени)', 'Исключено (шт)': n_t_dedup, 'Осталось (шт)': len(df_t_step1), 'Доля от исх.': f"{len(df_t_step1) / n_t_raw:.2%}"},
    {'Шаг фильтрации': '2. Отсечение дат вне периода ТЗ', 'Исключено (шт)': n_t_period, 'Осталось (шт)': len(df_t_step2), 'Доля от исх.': f"{len(df_t_step2) / n_t_raw:.2%}"},
    {'Шаг фильтрации': '3. Исключение операций без клиентов (сироты)', 'Исключено (шт)': n_t_orphans, 'Осталось (шт)': len(clean_transactions), 'Доля от исх.': f"{len(clean_transactions) / n_t_raw:.2%}"}
])

print("\nВОРОНКА ФИЛЬТРАЦИИ ТРАНЗАКЦИЙ (TRANSACTIONS):")
display(funnel_tx)

print(f"\nИТОГ ПРЕПРОЦЕССИНГА: Чистых клиентов — {len(clean_clients):,} | Чистых операций — {len(clean_transactions):,}")

ВОРОНКА ФИЛЬТРАЦИИ КЛИЕНТОВ (CLIENTS):


,Шаг фильтрации,Исключено (шт),Осталось (шт),Доля от исх.
0,0. Исходная база клиентов,0,5670,100.00%
1,1. Удаление полных дубликатов,40,5630,99.29%
2,2. Исключение конфликтующих ID,30,5600,98.77%



ВОРОНКА ФИЛЬТРАЦИИ ТРАНЗАКЦИЙ (TRANSACTIONS):


,Шаг фильтрации,Исключено (шт),Осталось (шт),Доля от исх.
0,0. Исходная база операций,0,619916,100.00%
1,1. Дедупликация txn_id (с сохранением времени),1500,618416,99.76%
2,2. Отсечение дат вне периода ТЗ,82,618334,99.74%
3,3. Исключение операций без клиентов (сироты),32115,586219,94.56%



ИТОГ ПРЕПРОЦЕССИНГА: Чистых клиентов — 5,600 | Чистых операций — 586,219


## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [69]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

In [70]:
cleaning_log = pd.DataFrame(log_records, columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

print(f"Всего зафиксировано решений: {len(cleaning_log)} шт.\n")
display(cleaning_log)

Всего зафиксировано решений: 10 шт.



,field,issue,scope,n_rows,value_before,action,value_after,rationale
0,monthly_income,системный маркер пропуска данных (-999),клиенты со значением -999,56,-999.00,исправлено,NaN,"технический маркер пропущенного значения, заме..."
1,city,"разнобой написания (регистр, префиксы г., сокр...",все строки clients,1678,"17 вариантов написания (г. Москва, СПб, казань...",исправлено,6 канонических городов РФ,сведено к единому классификатору 6 городов РФ ...
2,category,12 вариантов вместо 9 по ТЗ (разнобой регистра...,все строки transactions,51302,"12 категорий (Кафе/рестораны, Связь и Интернет...",исправлено,9 категорий,приведено к 9 регламентированным категориям ТЗ...
3,monthly_income,"значения выше верхней границы МКР (>197,050 ₽)","клиенты с доходом > 197,050 ₽",380,"диапазон до 1,052,900.00 ₽",оставлено,"диапазон до 1,052,900.00 ₽",96% выбросов относятся к сегментам Премиальный...
4,amount,"суммы операций выше границы МКР (>5,014 ₽)","операции с суммой > 5,014 ₽",50338,"диапазон до 152,959.72 ₽",оставлено,"диапазон до 152,959.72 ₽","крупные покупки (электроника, путешествия) явл..."
5,all (строки clients),полные дубликаты записей,все клиенты,40,полные клоны строк,удалено,уникальные строки,устранение технического сбоя повторной выгрузки
6,client_id,коллизии первичного ключа (конфликт атрибутов ...,"client_id: C005196, C002745, C000221 и др.",30,30 строк с противоречивыми данными,удалено,ключи исключены,невозможно достоверно определить истинные данн...
7,txn_id,дубликаты txn_id (полные клоны и записи с обре...,все транзакции,1500,повторяющиеся txn_id,удалено,уникальные txn_id с полным временем,"сохранена более детальная запись с секундами, ..."
8,txn_datetime,операции вне периода ТЗ (<2024-01-01 и >2025-0...,транзакции конца 2023 г. и 2025-2026 гг.,82,2023-11 .. 2026-07,ограничено,2024-01-01 .. 2025-06-30,строгое соответствие границам анализа по регла...
9,client_id (transactions),транзакции без профиля в clients (сироты),операции по неизвестным ID,32115,транзакции без клиента,удалено,только клиенты из чистой базы,"клиенты не идентифицированы, согласие на марке..."


## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [66]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")